# 07 - CNN 1D para Clasificacion de Fallas

## Objetivo
Clasificar el tipo de falla (4 clases) usando Convoluciones 1D.

## Por que CNN 1D?
- Detecta patrones LOCALES en la secuencia temporal
- Mas rapida de entrenar que LSTM
- Cada tipo de falla tiene una "forma" distinta que los filtros convolucionales pueden capturar

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from src.data.preprocessor import FiberDataPreprocessor
from src.data.dataset import create_dataloaders
from src.models.cnn1d_classifier import CNN1DClassifier
from src.utils.config_loader import load_config, get_path
from src.utils.visualization import apply_style, plot_training_history, plot_confusion_matrix

apply_style()
config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Preparar datos
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

feature_cols = ['optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db',
                'chromatic_dispersion', 'temperature', 'humidity', 'log_ber']
feature_cols = [c for c in feature_cols if c in train_df.columns]

cfg = config['models']['cnn1d']
seq_len = config['preprocessing']['sequence_length']

# DataLoaders con fault_label como target (4 clases)
train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    feature_columns=feature_cols,
    target_column='fault_label',
    sequence_length=seq_len,
    batch_size=cfg['batch_size'],
    dataset_type='sequence',
)

In [ ]:
# Crear modelo
model = CNN1DClassifier(
    n_features=len(feature_cols),
    channels=cfg['channels'],
    kernel_size=cfg['kernel_size'],
    num_classes=cfg['num_classes'],
    dropout=cfg['dropout'],
).to(device)

criterion = nn.CrossEntropyLoss()  # Loss para clasificacion multi-clase
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

In [ ]:
# Training Loop
epochs = cfg['epochs']
train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.long().to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.long().to(device)
            output = model(batch_x)
            val_loss += criterion(output, batch_y).item()
    
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), str(get_path('models') / 'cnn1d_best.pt'))
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')
    
    if patience_counter >= 10:
        print(f'Early stopping en epoch {epoch+1}')
        break

In [ ]:
# Curvas y evaluacion
fig = plot_training_history(train_losses, val_losses)
plt.show()

# Evaluacion en test
model.load_state_dict(torch.load(str(get_path('models') / 'cnn1d_best.pt'), weights_only=True))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        output = model(batch_x)
        preds = output.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(batch_y.numpy())

class_names = ['normal', 'physical_cut', 'degradation', 'bad_splice']
print('Classification Report:')
print(classification_report(all_targets, all_preds, target_names=class_names))

cm = confusion_matrix(all_targets, all_preds)
fig = plot_confusion_matrix(cm, class_names, title='CNN 1D - Matriz de Confusion')
plt.show()